# NAS Search — ResNet-56 / CIFAR-100
Loads accumulated NAS samples from `nas_data/`, trains an MLP accuracy estimator, then runs evolutionary search for two compression objectives:
- **DMC-Ultra**: minimise weight storage subject to fitting on the ATmega2560 (Arduino Mega).
- **DMC-Board**: maximise accuracy subject to the same hardware constraint.


In [ ]:
import os, sys, json, torch
import matplotlib.pyplot as plt

_HERE      = os.path.dirname(os.path.abspath('.'))
_PROJ_ROOT = os.path.abspath(os.path.join('.', '../../..'))
_SHARED    = os.path.abspath(os.path.join('.', '../shared'))
sys.path.insert(0, _PROJ_ROOT)
sys.path.insert(0, _SHARED)

from development import (
    ConfigEncoder, evolutionary_search_compression_config,
    QuantizationScheme,
)
from development.experiments.resnet   import get_model
from development.experiments.cifar100 import get_data_loaders, get_metric
from hardware_specs import ATMEGA2560, make_ultra_condition, make_board_condition
from nas_utils      import load_nas_data, train_estimator, make_estimate_fn


In [ ]:
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED        = 25
INPUT_SHAPE = (3, 32, 32)
HARDWARE    = ATMEGA2560
NAS_DATA_DIR = 'nas_data'
MODELS_DIR   = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

torch.manual_seed(SEED)
print(f'Device: {DEVICE}')


## 1 — Load & aggregate NAS samples

In [ ]:
nas_data = load_nas_data(NAS_DATA_DIR)
print('Keys:', list(nas_data.keys())[:4], '...')


## 2 — Build baseline model and ConfigEncoder

In [ ]:
print('Loading model …')
baseline_model = get_model().to(DEVICE)
nas_encoder    = ConfigEncoder(baseline_model)


## 3 — Encode NAS data

In [ ]:
encoded_data = nas_encoder.encode(nas_data, with_metric=True)
print(f'Encoded shape: {encoded_data.shape}   (rows=samples, cols=features+1 metric)')


## 4 — Train accuracy estimator

In [ ]:
print('Training estimator …')
estimator, x_mu, x_std, y_mu, y_std, history = train_estimator(
    encoded_data,
    device     = DEVICE,
    hidden_dim = 256,
    dropout    = 0.2,
    epochs     = 2000,
    batch_size = 128,
    val_split  = 0.2,
    seed       = SEED,
)


## 5 — Estimator training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'][50:], label='train')
axes[0].plot(history['val_loss'][50:],   label='val')
axes[0].set_title('Huber loss'); axes[0].legend()

axes[1].plot(history['train_mae'][50:], label='train')
axes[1].plot(history['val_mae'][50:],   label='val')
axes[1].set_title('MAE (accuracy %)'); axes[1].legend()
plt.tight_layout(); plt.show()

print(f'Final val MAE: {history["val_mae"][-1]:.3f}%')


In [ ]:
# Predicted vs actual on the full encoded dataset
import torch
X_all = ((encoded_data[:, :-1].float() - x_mu) / x_std).to(DEVICE)
with torch.no_grad():
    Y_pred = (estimator(X_all) * y_std.to(DEVICE) + y_mu.to(DEVICE)).cpu().numpy()
Y_true = encoded_data[:, -1].numpy()

plt.figure(figsize=(5, 5))
plt.scatter(Y_true, Y_pred, s=8, alpha=0.6)
lims = [min(Y_true.min(), Y_pred.min()), max(Y_true.max(), Y_pred.max())]
plt.plot(lims, lims, 'r--', lw=1)
plt.xlabel('True accuracy (%)'); plt.ylabel('Predicted accuracy (%)')
plt.title('Estimator calibration'); plt.tight_layout(); plt.show()


## 6 — Evolutionary search

In [ ]:
train_loader, test_loader = get_data_loaders()
calibration_data = next(iter(train_loader))[0].to(DEVICE)

metric_fn = get_metric()
baseline_acc = baseline_model.fuse(device=DEVICE).evaluate(
    test_loader, {'acc': metric_fn}, device=DEVICE
)['acc']
print(f'Baseline accuracy: {baseline_acc:.2f}%')

estimate = make_estimate_fn(estimator, nas_encoder, x_mu, x_std, y_mu, y_std, DEVICE)
original_size = baseline_model.fuse(device=DEVICE).get_size_in_bytes()


### DMC-Ultra — minimise weight storage

In [ ]:
ultra_condition = make_ultra_condition(HARDWARE)

best_ultra_raw, ultra_info = evolutionary_search_compression_config(
    baseline_model,
    estimate,
    INPUT_SHAPE,
    calibration_data,
    condition  = ultra_condition,
    objective  = lambda metric, size, ram, cfg: size,
    maximize   = False,
    verbose    = True,
    population_size = 75,
    generations     = 75,
)

ultra_config = baseline_model.decode_compression_dict_hyperparameter(best_ultra_raw)
print('\nDMC-Ultra config:')
print(json.dumps({k: str(v) for k, v in ultra_config.items()}, indent=2))
print('Search info:', ultra_info)


### DMC-Board — maximise accuracy within board constraints

In [ ]:
board_condition = make_board_condition(HARDWARE, baseline_acc, max_drop=5.0)

best_board_raw, board_info = evolutionary_search_compression_config(
    baseline_model,
    estimate,
    INPUT_SHAPE,
    calibration_data,
    condition  = board_condition,
    objective  = lambda metric, size, ram, cfg: metric,
    maximize   = True,
    verbose    = True,
    population_size = 75,
    generations     = 75,
)

board_config = baseline_model.decode_compression_dict_hyperparameter(best_board_raw)
print('\nDMC-Board config:')
print(json.dumps({k: str(v) for k, v in board_config.items()}, indent=2))
print('Search info:', board_info)


## 7 — Save configs

In [ ]:
torch.save(ultra_config, os.path.join(MODELS_DIR, 'dmc_ultra_config.pth'))
torch.save(board_config, os.path.join(MODELS_DIR, 'dmc_board_config.pth'))
print('Configs saved to models/')
